<div style="background-color:#EAEAEA;padding:20px;border-left:5px solid #6C757D;border-radius:6px;">
<table style="width:100%; border:none;"><tr style="border:none;">
<td style="border:none; vertical-align:top;">
<h1 style="font-size:32px; margin-top:0;">Master's Thesis</h1>
<hr style="margin:16px 0 22px 0;">
<p style="font-size:22px; line-height:1.5; margin:0;"><strong>Master's Degree in Advanced Physics</strong> - <strong>Universitat de Valencia</strong></p>
<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:</p>
<div style="font-size:25px;font-weight:700;line-height:1.3;margin-top:14px;margin-bottom:26px;">Fast Simulation of Neutrino Oscillations in Matter</div>
<p style="font-size:14px; line-height:1.55;"><strong>Author</strong><br>Juan Ramon Diaz Santos - <a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a></p>
<p style="font-size:14px; line-height:1.55;"><strong>Supervisors</strong><br>Roberto Ruiz de Austri Bazan - <a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>Michele Lucente - <a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a></p>
<p style="font-size:14px; line-height:1.55; margin-bottom:0;"><strong>Date</strong><br>September 2026</p></td>
<td style="border:none;width:230px;padding-left:25px;text-align:right;vertical-align:top;"><img src="../../logo_uv.png" alt="Universitat de Valencia" style="width:200px; margin-top:5px;"></td>
</tr></table></div>

# Intrinsic Validation 0: Summary
---
This notebook aggregates the internal-consistency results produced by `intrinsic1_StandarModel.ipynb` through `intrinsic6_RuntimeContext.ipynb` and presents a compact dashboard of `tpeanuts`'s own internal validation: numerical-vs-perturbative agreement, numerical convergence, closed-form eigenvalue solvers, adiabatic Landau-Zener option, and float32/float64 and CPU/CUDA runtime consistency. Run notebooks 1-6 first to generate the required CSV files.

Unlike `nusquids0_summary.ipynb`/`validation_legacy0_summary.ipynb`, this dashboard never depends on an external reference code (no PEANUTS, no nuSQuIDS): every comparison here is `tpeanuts` checked against itself under a different method, precision, or device, so no availability guard is needed.

## Table of Contents

| # | Section |
|---|---|
| [0](#0.-Theoretical-Framework) | **Theoretical Framework** |
| [1](#1.-Libraries) | **Libraries** |
| [2](#2.-Paths-and-Configuration) | **Paths and Configuration** |
| [3](#3.-Load-Comparison-CSVs) | **Load Comparison CSVs** |
| [4](#4.-Global-Aggregation) | **Global Aggregation** |
| [5](#5.-Visualisation-and-Export) | **Visualisation and Export** |
| [6](#6.-Summary) | **Summary** |

## 0. Theoretical Framework
---
This section records the scope and provenance of the notebook.

**References**

No external references are cited in this notebook.

## 1. Libraries


In [17]:
from __future__ import annotations

import re

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tpeanuts.notebooks.notebookConfig import load_notebook_config
from tpeanuts.notebooks.notebooks_helper import (
    TOL_PPB,
    TOL_PPM,
    save_and_show,
    status_from_rel,
)


## 2. Paths and Configuration

`load_notebook_config()` resolves the repository root and the shared output directory. All CSV files from notebooks 1-6 are written to the same flat directory `validation/intrinsic/`. This notebook reads from that directory and writes its own exports to a `summary/` subdirectory inside it.

Each per-case summary CSV is named `validation_intrinsic{N}_..._summary.csv`, identifying which `intrinsicN_*.ipynb` notebook produced it. As in `nusquids0_summary.ipynb`, the prefix-to-label table below is built by scanning the sibling `intrinsic1_*.ipynb` .. `intrinsic6_*.ipynb` filenames directly rather than being hardcoded, so it stays correct across renames: the label for notebook `N` is whatever follows `intrinsicN_` in that notebook's filename.

### 2.1 Paths

Repository-relative input and output locations are resolved here, ensuring reproducible execution without hidden state or external notebook dependencies.

### 2.2 Configuration

Physical parameters, numerical grids, precision, runtime context, and validation tolerances are fixed here for every subsequent calculation.

In [18]:
config          = load_notebook_config()
VALIDATION_ROOT = config.output_dir("validation", "intrinsic")
OUTPUT_DIR      = VALIDATION_ROOT / "summary"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SHOW_PLOTS      = config.show_plots

INTRINSIC_NB_DIR = config.package_dir / "notebooks" / "validation" / "intrinsic"
PREFIX_TO_LABEL = {}
for nb_path in sorted(INTRINSIC_NB_DIR.glob("intrinsic[1-6]_*.ipynb")):
    m = re.match(r"intrinsic([1-6])_(.+)\.ipynb$", nb_path.name)
    if m:
        PREFIX_TO_LABEL[m.group(1)] = m.group(2)

print(f"Validation root : {VALIDATION_ROOT}")
print(f"Output dir      : {OUTPUT_DIR}")
print(f"Notebook -> label : {PREFIX_TO_LABEL}")


Validation root : v:\output\validation\intrinsic
Output dir      : v:\output\validation\intrinsic\summary
Notebook -> label : {'1': 'StandarModel', '2': 'BSM_extension_NSI', '3': 'BSM_extension_sterile', '4': 'Numerical_Method', '5': 'Perturbative_Method', '6': 'RuntimeContext'}


## 3. Load Comparison CSVs

Discovers all `validation_intrinsic*_summary.csv` files in the validation directory. These files are produced by `validation_metrics`/`validation_summary_plot` in notebooks 1-6 and contain one row per internal comparison ("case"), each with `max_abs`, `mean_abs`, `max_rel`, and `mean_rel` columns. Each file's rows are labelled with their source notebook via `PREFIX_TO_LABEL` (Section 2), extracted from the filename with a `validation_intrinsic(\d)_` regex so it does not depend on the topic suffix some files add (`..._numerical_summary.csv`, `..._sm_summary.csv`, etc.) and some omit.

In [19]:
FILENAME_RE = re.compile(r"validation_intrinsic([1-6])_")

case_rows = []
for path in sorted(VALIDATION_ROOT.glob("validation_intrinsic*_summary.csv")):
    m = FILENAME_RE.match(path.name)
    notebook_num = m.group(1) if m else "?"
    notebook_label = PREFIX_TO_LABEL.get(notebook_num, "unknown")
    df = pd.read_csv(path)
    if "case" not in df.columns or not {"max_abs", "mean_abs", "max_rel", "mean_rel"}.issubset(df.columns):
        continue
    for _, row in df.iterrows():
        case_rows.append({
            "notebook": notebook_label,
            "file": path.name,
            "case": row["case"],
            "max_abs": row["max_abs"],
            "mean_abs": row["mean_abs"],
            "max_rel": row["max_rel"],
            "mean_rel": row["mean_rel"],
        })

case_summary = pd.DataFrame(case_rows)

if case_summary.empty:
    print("No comparison CSVs found. Run intrinsic1-6 notebooks first.")
else:
    display(case_summary.sort_values(["notebook", "max_rel"], ascending=[True, False]))


,notebook,file,case,max_abs,mean_abs,max_rel,mean_rel
9,BSM_extension_NSI,validation_intrinsic2_nsi_summary.csv,atmosphere production to detector,2.824929e-02,3.916036e-04,6.380544e-01,3.425123e-03
6,BSM_extension_NSI,validation_intrinsic2_nsi_summary.csv,pure flavour through Earth,2.770251e-02,7.025787e-04,5.647736e-01,5.428697e-03
7,BSM_extension_NSI,validation_intrinsic2_nsi_summary.csv,solar production to detector,1.030304e-02,4.263785e-04,3.411535e-02,1.295173e-03
8,BSM_extension_NSI,validation_intrinsic2_nsi_summary.csv,atmosphere production to surface,1.651401e-11,2.198471e-13,1.551311e-09,1.615926e-11
13,BSM_extension_sterile,validation_intrinsic3_sterile_summary.csv,atmosphere production to detector,2.294253e-02,2.186725e-04,4.875472e-01,2.621386e-03
10,BSM_extension_sterile,validation_intrinsic3_sterile_summary.csv,pure flavour through Earth,2.327787e-02,3.890716e-04,4.712319e-01,4.022859e-03
11,BSM_extension_sterile,validation_intrinsic3_sterile_summary.csv,solar production to detector,8.599296e-03,2.271398e-04,2.556263e-02,7.682838e-04
12,BSM_extension_sterile,validation_intrinsic3_sterile_summary.csv,atmosphere production to surface,7.388379e-09,2.539384e-10,1.124517e-07,2.892111e-09
16,Numerical_Method,validation_intrinsic4_numerical_summary.csv,3+1 sterile: 928 steps,3.219377e-03,1.266977e-04,6.249419e-02,1.150754e-03
15,Numerical_Method,validation_intrinsic4_numerical_summary.csv,NSI: 928 steps,4.966295e-03,1.474261e-04,5.913994e-02,1.688326e-03


## 4. Global Aggregation

Groups the per-case summary by source notebook (`intrinsic1` .. `intrinsic6`) and reports the overall worst-case and typical internal-consistency errors for each. The `status` column is set by `status_from_rel`: `PASS < ppb` (< 10⁻⁹), `PASS < ppm` (< 10⁻⁶), `CHECK < 1e-3` (< 10⁻³), or `FAIL` otherwise.

**Expected results:**<br>
All six notebooks are internal `tpeanuts`-vs-`tpeanuts` self-consistency checks (different method, precision, or device), so most rows are expected at `PASS < ppb`/`PASS < ppm`; genuine method-level approximations (numerical-step convergence, perturbative truncation) are expected to sit at the `CHECK` level until the compared setting is tightened.

In [ ]:
if not case_summary.empty:
    global_summary = case_summary.groupby("notebook", as_index=False).agg(
        cases=("case", "count"),
        max_abs_err=("max_abs", "max"),
        mean_abs_err=("mean_abs", "mean"),
        median_abs_err=("max_abs", "median"),
        max_rel_err=("max_rel", "max"),
        mean_rel_err=("mean_rel", "mean"),
        median_rel_err=("max_rel", "median"),
    )
    global_summary["status"] = global_summary["max_rel_err"].map(status_from_rel)
    display(global_summary.sort_values("max_rel_err", ascending=False))
else:
    global_summary = pd.DataFrame()
    print("No data available for aggregation.")


## 5. Visualisation and Export

Single grouped bar chart comparing maximum and mean absolute and relative errors across all six intrinsic-validation notebooks on a logarithmic scale, one group of bars per notebook, absolute errors in one colour and relative errors in another (lighter shade for the mean), matching `nusquids0_summary.ipynb`'s and `validation_legacy0_summary.ipynb`'s layout. Horizontal reference lines mark the 1 ppm (10⁻⁶) and 1 ppb (10⁻⁹) tolerance levels.

Two aggregate CSV files are exported:

- `intrinsic0_case_summary.csv` -- one row per individual comparison case with its notebook label.
- `intrinsic0_global_summary.csv` -- one row per notebook with aggregate worst-case and typical metrics.

In [ ]:
if not global_summary.empty:
    ordered = global_summary.sort_values("max_rel_err", ascending=True)
    x = np.arange(len(ordered))
    width = 0.2
    fig, ax = plt.subplots(figsize=(9.0, 5.5))
    ax.bar(x - 1.5 * width, ordered["max_abs_err"], width, color="C0", label="max abs error")
    ax.bar(x - 0.5 * width, ordered["mean_abs_err"], width, color="C0", alpha=0.5, label="mean abs error")
    ax.bar(x + 0.5 * width, ordered["max_rel_err"], width, color="C1", label="max rel error")
    ax.bar(x + 1.5 * width, ordered["mean_rel_err"], width, color="C1", alpha=0.5, label="mean rel error")
    ax.set_yscale("log")
    ax.axhline(TOL_PPM, color="dimgray", ls="--", label="1 ppm")
    ax.axhline(TOL_PPB, color="lightgray", ls=":", label="1 ppb")
    ax.set_xticks(x)
    ax.set_xticklabels(ordered["notebook"], rotation=25, ha="right")
    ax.set_ylabel("error")
    ax.set_title("Intrinsic validation: absolute and relative errors")
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    save_and_show("intrinsic0_summary.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)
    display(ordered[["notebook", "cases", "max_abs_err", "mean_abs_err", "median_abs_err", "max_rel_err", "mean_rel_err", "median_rel_err", "status"]])
    case_summary.to_csv(OUTPUT_DIR / "intrinsic0_case_summary.csv", index=False)
    global_summary.to_csv(OUTPUT_DIR / "intrinsic0_global_summary.csv", index=False)
    print("Exported intrinsic0_case_summary.csv and intrinsic0_global_summary.csv")
else:
    print("No comparison CSV files found. Run intrinsic1-6 notebooks first.")


## 6. Summary
---
This notebook documented **Intrinsic Validation 0: Summary**, including its methodology, principal calculations and resulting diagnostics. The preceding sections contain the detailed numerical outputs and visual checks needed to interpret the result.


### 6.1 Standard Model: Numerical/Perturbative and Runtime-Option Comparison

This table restricts `case_summary` (Section 3) to the Standard Model and groups it by comparison category: the solar-surface propagation checks from Section 3 of `intrinsic1_StandarModel.ipynb` (`adiabatic_approximated` and `adiabatic_exact`, each versus `numerical`), the two further end-to-end numerical-versus-perturbative propagation checks from `intrinsic1_StandarModel.ipynb` (solar neutrinos to the detector, Section 5; atmosphere to the underground detector, Section 6.2), the perturbative-option checks from Section 3 of `intrinsic5_Perturbative_Method.ipynb` (reunitarization on/off, Cardano versus `torch.linalg.eigvalsh` eigenvalues, solar Landau-Zener at the solar surface), and the runtime checks from Section 3 of `intrinsic6_RuntimeContext.ipynb` (numerical float32 versus numerical float64, numerical CPU versus numerical CUDA).

**Note:** `intrinsic1` uses a relative-error threshold of $10^{-5}$ while `intrinsic5`/`intrinsic6` use $10^{-3}$ (see each notebook's Section 0.7/2.2), so `max_rel`/`mean_rel` are directly comparable only within rows sharing the same `notebook` column; `max_abs`/`mean_abs` remain comparable across all rows.

**Expected results:**<br>
- Every row's four error metrics are finite.
- The reunitarization, eigenvalue-solver, Landau-Zener, precision and device rows should not exceed the propagation-method errors by orders of magnitude.


In [22]:
STANDARD_MODEL_PROPAGATION_CATEGORY = {'solar surface: adiabatic_approximated vs numerical': 'Solar Propagation: adiabatic_approximated vs numerical', 'solar surface: adiabatic_exact vs numerical': 'Solar Propagation: adiabatic_exact vs numerical', 'solar production to detector': 'Numerical vs perturbative (solar to detector)', 'atmosphere production to detector': 'Numerical vs perturbative (atmosphere to detector)'}
STANDARD_MODEL_OPTION_CATEGORY = {'reunitarize=False': 'Reunitarize off vs numerical', 'reunitarize=True': 'Reunitarize on vs numerical', 'Cardano vs torch': 'Analytical eigenvalues: Cardano vs torch', 'LZ at solar surface': 'Landau-Zener (solar surface)', 'numerical float32 vs numerical float64': 'Precision: numerical float32', 'numerical CPU vs numerical CUDA': 'Device: numerical CPU'}

def standard_model_category(row):
    if row['notebook'] == 'StandarModel':
        return STANDARD_MODEL_PROPAGATION_CATEGORY.get(row['case'])
    if row['case'].startswith('Standard Model: '):
        return STANDARD_MODEL_OPTION_CATEGORY.get(row['case'].split('Standard Model: ', 1)[1])
    return None

if case_summary.empty:
    print('No comparison CSV files found. Run intrinsic1-6 notebooks first.')
else:
    standard_model_summary = case_summary.assign(category=case_summary.apply(standard_model_category, axis=1)).dropna(subset=['category']).set_index('case')[['notebook', 'category', 'max_abs', 'mean_abs', 'max_rel', 'mean_rel']]
    display(standard_model_summary.style.format({column: '{:.3e}' for column in standard_model_summary.select_dtypes(include='number').columns}))
    standard_model_summary.to_csv(OUTPUT_DIR / 'intrinsic0_standard_model_master_summary.csv')
    assert np.isfinite(standard_model_summary[['max_abs', 'mean_abs', 'max_rel', 'mean_rel']].to_numpy()).all()
    print('Standard Model cross-notebook comparison completed successfully.')


,notebook,category,max_abs,mean_abs,max_rel,mean_rel
case,,,,,,
solar surface: adiabatic_approximated vs numerical,StandarModel,Solar Propagation: adiabatic_approximated vs numerical,4.879e-03,3.846e-04,4.576e-02,2.502e-03
solar surface: adiabatic_exact vs numerical,StandarModel,Solar Propagation: adiabatic_exact vs numerical,4.873e-03,3.797e-04,4.564e-02,2.470e-03
solar production to detector,StandarModel,Numerical vs perturbative (solar to detector),3.224e-03,1.320e-04,9.245e-03,3.960e-04
atmosphere production to detector,StandarModel,Numerical vs perturbative (atmosphere to detector),9.769e-03,1.567e-04,9.548e-01,2.007e-03
Standard Model: reunitarize=False,Perturbative_Method,Reunitarize off vs numerical,1.600e-02,1.376e-03,1.558e-01,7.666e-03
Standard Model: reunitarize=True,Perturbative_Method,Reunitarize on vs numerical,8.774e-03,8.619e-04,1.460e-01,6.248e-03
Standard Model: Cardano vs torch,Perturbative_Method,Analytical eigenvalues: Cardano vs torch,9.065e-13,1.558e-14,1.634e-11,1.187e-13
Standard Model: LZ at solar surface,Perturbative_Method,Landau-Zener (solar surface),0.000e+00,0.000e+00,0.000e+00,0.000e+00
Standard Model: numerical float32 vs numerical float64,RuntimeContext,Precision: numerical float32,1.036e-04,1.115e-05,1.747e-04,3.350e-05


Standard Model cross-notebook comparison completed successfully.
